**Mini-projet : Intégration de l’IA MCP + Agents dans Gemin**

**Objectif :** Créer une application multi-agents complète qui orchestre plusieurs serveurs MCP à l’aide d’un LLM ( Gemini ) et d’une politique flexible pilotée par outil (le LLM détermine les prochaines étapes, et non des flux prédéfinis).
Environnement cible : Google Colab (obligatoire) (possibilité d’exécution locale).



Thème de projet suggéré (choisissez-en un)
Choisissez un flux de travail réaliste qui tire parti de plusieurs outils. Exemples :

« Assistant d'espace de travail » (fichiers + git + résumé personnalisé)
« Assistant de recherche » (fichiers + outil de citation personnalisé + outil de mise en forme)
« Assistant de développement » (git + système de fichiers + wrapper d'exécution de tests/linter personnalisé)
« Assistant de café » (système de fichiers + menu/tarification personnalisés + historique Git des commandes)
Vous pouvez choisir n'importe quel autre thème, pourvu qu'il assure une cohérence d'affichage sur l'ensemble des serveurs.






##

##



**Intégrer des serveurs MCP tiers (Gemini)**


1. Configuration de Colab : installer les dépendances
Créez un nouveau notebook Colab et ajoutez cette cellule :

In [1]:
%pip install -qU \
  "langchain>=0.3" \
  "langgraph>=0.2" \
  "langchain-google-genai>=2.0" \
  "google-genai>=1.0" \
  "langchain-mcp-adapters==0.2.1" \
  "nest_asyncio"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.9/136.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.5/247.5 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.9 MB/s eta 0:00:00


##

##

**Notes**

langchain-google-genai est l'intégration LangChain pour Gemini.
langchain-mcp-adapters fournit un client MCP compatible avec les outils LangChain.


2. Configurer GOOGLE_API_KEY dans Colab

Pour utiliser les modèles Gemini, vous devez définir la GOOGLE_API_KEYvariable d'environnement dans Colab.



3. Vérifier la disponibilité du nœud/NPM (requis pour de nombreux serveurs MCP)

De nombreux serveurs MCP sont distribués sous forme de packages Node exécutables via npx.

In [2]:
!node --version
!npx --version

v20.19.0
10.8.2


En cas de manque :

In [3]:
!apt-get -qq update
!apt-get -qq install -y nodejs npm
!node --version
!npx --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package gyp.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../000-gyp_0.1+20210831gitd6c5dd5-5_all.deb ...
Unpacking gyp (0.1+20210831gitd6c5dd5-5) ...
Selecting previously unselected package javascript-common.
Preparing to unpack .../001-javascript-common_11+nmu1_all.deb ...
Unpacking javascript-common (11+nmu1) ...
Selecting previously unselected package libjs-events.
Preparing to unpack .../002-libjs-events_3.3.0+~3.0.0-2_all.deb ...
Unpacking libjs-events (3.3.0+~3.0.0-2) ...
Selecting previously unselected package libjs-highlight.js.
Preparing to unpack .../003-libjs-highlight.js_9.18.5+dfsg1-1_all.deb ...
Unpacking libjs-highlight.js (9.18.5+dfsg1-1) ...
Selecting previously 

import asyncio
import os
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent



In [4]:
import asyncio
import os
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent



7. Créez un agent Gemini capable d'utiliser ces outils


8. Implémenter un serveur MCP personnalisé en Python

Utilisez FastMCP (serveur stdio simple) :

In [5]:
%pip install -qU "fastmcp>=2.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 765.2/765.2 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.1/230.1 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.0/170.0 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.0/273.0 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 35.1 MB/s eta 0:00:00


Créer un fichier serveur (exemples d'outils) :

In [31]:
from pathlib import Path
import textwrap

raw_server_code_string = """
    from fastmcp import FastMCP
    from typing import Dict, List

    mcp = FastMCP(name='custom_ops')

    @mcp.tool
    def ping() -> str:
        '''Health check tool.'''
        return 'pong'

    @mcp.tool
    def summarize_lines(lines: List[str]) -> Dict[str, int]:
        '''Example tool: return counts about a list of lines.'''
        total = len(lines)
        nonempty = sum(1 for l in lines if l.strip())
        return {'total_lines': total, 'nonempty_lines': nonempty}

    if __name__ == '__main__':
        # Utilisation du transport sse (HTTP) pour la compatibilité Colab
        mcp.run(transport='sse', port=12347)
"""
server_code = textwrap.dedent(raw_server_code_string)
server_path = Path('/content/custom_mcp_server.py')
server_path.write_text(server_code, encoding='utf-8')
print('Serveur prêt (SSE/HTTP mode):', server_path)

Serveur prêt (SSE/HTTP mode): /content/custom_mcp_server.py


Vous pouvez utiliser tous les outils pertinents pour le thème de votre projet. Exemples :

price_order(items),get_menu()
extract_entities(text),validate_schema(data)
format_markdown(text),generate_changelog(diff)


9. Ajoutez le serveur personnalisé à votre client MCP

In [32]:
import asyncio
import nest_asyncio
import subprocess
import time
from langchain_mcp_adapters.client import MultiServerMCPClient

nest_asyncio.apply()

# Nettoyage et démarrage du serveur SSE
!fuser -k 12347/tcp
server_proc = subprocess.Popen(['python', '/content/custom_mcp_server.py'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5) # Attente du démarrage

mcp_connections = {
    "custom_ops": {
        "transport": "sse",
        "url": "http://127.0.0.1:12347/sse",
    }
}

async def connect():
    client = MultiServerMCPClient(mcp_connections, tool_name_prefix=True)
    return await client.get_tools()

try:
    tools2 = asyncio.get_event_loop().run_until_complete(connect())
    print(f"Succès ! {len(tools2)} outils chargés.")
    print("Outils :", [t.name for t in tools2])
except Exception as e:
    print(f"Erreur : {e}")

Succès ! 2 outils chargés.
Outils : ['custom_ops_ping', 'custom_ops_summarize_lines']


## Thème choisi : Assistant de recherche

Flux de travail :
- **Serveur 1 : Filesystem** → lire des fichiers de notes/recherche (`@modelcontextprotocol/server-filesystem`)
- **Serveur 2 : Git** → historique des versions du dossier de recherche (`mcp_server_git`)
- **Serveur 3 : Custom Ops** → outils personnalisés `summarize_notes()` et `extract_citations()` pour structurer les résultats

L’agent Gemini décide lui-même quel outil appeler selon la question (policy-driven tool use).


## Assistant de recherche

Thème sélectionné : **Assistant de recherche**

Ce projet orchestre 3 serveurs MCP :
- **Filesystem** : accès aux fichiers de notes/recherche
- **Git** : historique des versions du dossier
- **Custom Ops** : outils `summarize_notes()` et `extract_citations()`

L’agent Gemini choisit dynamiquement les outils à appeler (policy-driven).


In [33]:
from pathlib import Path
import textwrap

server_path = Path("/content/custom_ops_server.py")
server_path.write_text(textwrap.dedent("""
    from fastmcp import FastMCP
    from typing import Dict, List

    mcp = FastMCP(name='research_ops')

    @mcp.tool
    def summarize_notes(notes: List[str]) -> Dict[str, str]:
        \"\"\"Synthèse de notes de recherche : résumé, thèmes, action_items.\"\"\"
        lines = [l.strip() for l in notes if l.strip()]
        return {
            'summary': ' | '.join(lines[:3]) if lines else 'No content',
            'total_notes': str(len(lines)),
            'themes': 'AI, Agents, MCP'
        }

    @mcp.tool
    def extract_citations(text: str) -> List[str]:
        \"\"\"Extrait des citations fictives d'un texte.\"\"\"
        import re
        matches = re.findall(r'\\[(.*?)\\]', text)
        return matches if matches else ['No citation found']

    if __name__ == '__main__':
        # Utilisation de SSE pour la compatibilité Colab sur un nouveau port
        mcp.run(transport='sse', port=12348)
"""), encoding="utf-8")

print("Serveur personnalisé créé (mode SSE) :", server_path)

Serveur personnalisé créé (mode SSE) : /content/custom_ops_server.py


In [37]:
import asyncio
import os
import subprocess
import time
import nest_asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient

nest_asyncio.apply()

WORKDIR = "/content/research_workspace"
os.makedirs(WORKDIR, exist_ok=True)

# Redémarrage du serveur personnalisé en mode SSE (Port 12348)
!fuser -k 12348/tcp
server_proc = subprocess.Popen(['python', '/content/custom_ops_server.py'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# Dans Colab, SSE est la méthode fiable pour les serveurs MCP Python.
# On se concentre sur research_ops pour éviter les erreurs de flux stdio.
mcp_connections = {
    "research_ops": {
        "transport": "sse",
        "url": "http://127.0.0.1:12348/sse",
    }
}

async def connect_servers():
    client = MultiServerMCPClient(mcp_connections, tool_name_prefix=True)
    return await client.get_tools()

try:
    tools = asyncio.get_event_loop().run_until_complete(connect_servers())
    print(f"Succès ! {len(tools)} outils chargés.")
    print("Outils disponibles :", [t.name for t in tools])
except Exception as e:
    print(f"Erreur de connexion : {e}")

12348/tcp:           10780
Succès ! 2 outils chargés.
Outils disponibles : ['research_ops_summarize_notes', 'research_ops_extract_citations']


In [39]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent
from google.colab import userdata

# Récupération de la clé API
try:
    api_key = userdata.get('GOOGLE_API_KEY')
except:
    api_key = None

# Initialiser le LLM Gemini
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0.1,
    google_api_key=api_key,
)

# Configuration de l'agent
# Note: 'prompt' remplace 'state_modifier' dans create_react_agent
system_prompt = (
    "Tu es un assistant de recherche intelligent. "
    "Tu utilises tes outils personnalisés pour synthétiser des informations. "
    "Réponds toujours en français."
)

agent_executor = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt
)

print("Agent de recherche prêt !")

Agent de recherche prêt !


/tmp/ipykernel_577/860786590.py:26: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(


In [40]:
queries = [
    "Utilise l'outil summarize_notes sur les lignes suivantes : ['L\'IA est puissante', 'MCP est utile'].",
    "Extrais les citations du texte suivant : '[source: 12] Le futur est agentique'.",
]

async def run_queries():
    for q in queries:
        print("\n" + "=" * 60)
        print("Question :", q)
        try:
            # Utilisation de l'agent_executor
            inputs = {"messages": [("user", q)]}
            async for event in agent_executor.astream(inputs, stream_mode="values"):
                message = event["messages"][-1]

            if message:
                print("Réponse :", message.content)
        except Exception as e:
            print("Erreur :", e)

# Exécution asynchrone dans le notebook
import asyncio
asyncio.get_event_loop().run_until_complete(run_queries())


Question : Utilise l'outil summarize_notes sur les lignes suivantes : ['L'IA est puissante', 'MCP est utile'].


Erreur : Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 6.292997666s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google